# MiniMax H3 — ComfyUI + Cloudflare (proven direct-launch path)

This notebook uses the exact process pattern that succeeded in the Cloudflare origin-isolation test.

Order:
1. Mount Drive
2. Check GPU
3. Clone/update repo
4. Install ComfyUI + required H3 custom nodes
5. Load Cloudflare tunnel secret
6. Start ComfyUI directly and prove `127.0.0.1:8188` is healthy
7. Start Cloudflare Tunnel directly and verify connector registration
8. Download H3 models while ComfyUI remains online
9. Restart **only ComfyUI** so the newly downloaded models appear

No workflow-restore section. No LTX section. No helper tunnel launcher.


## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PERSIST_MODELS_TO_DRIVE = False
PERSIST_OUTPUT_TO_DRIVE = True
DRIVE_ROOT = '/content/drive/MyDrive/MiniMax_H3_ComfyUI'


## 2. Check GPU

In [ ]:
import subprocess, os

def sh(cmd):
    return subprocess.check_output(cmd, shell=True, text=True).strip()

gpu_name = sh('nvidia-smi --query-gpu=name --format=csv,noheader | head -n1')
vram_mb = int(sh('nvidia-smi --query-gpu=memory.total --format=csv,noheader,nounits | head -n1'))
name = gpu_name.lower()
if 'rtx pro 6000' in name or 'blackwell' in name:
    H3_RESERVE_VRAM_GB = '6'
elif 'a100' in name:
    H3_RESERVE_VRAM_GB = '4'
elif 'l4' in name:
    H3_RESERVE_VRAM_GB = '2'
else:
    H3_RESERVE_VRAM_GB = '1'

print(f'GPU: {gpu_name} ({vram_mb/1024:.1f} GB)')
print('Reserved VRAM:', H3_RESERVE_VRAM_GB, 'GB')


## 3. Clone/update the H3 branch

In [ ]:
%cd /content
!rm -rf /content/All-testing /content/minimax_h3_comfy
!git clone --depth 1 --branch minimax-h3-colab https://github.com/Logan17de/All-testing.git /content/All-testing
!cp -r /content/All-testing/video/minimax_h3_comfy /content/minimax_h3_comfy
%cd /content/minimax_h3_comfy


## 4. Install/update ComfyUI + required H3 custom nodes

This installs the application and nodes only. The large H3 model downloads happen **after** the public ComfyUI server is verified.


In [ ]:
import os, subprocess, sys

os.environ['COMFY_ROOT'] = '/content/ComfyUI'
os.environ['H3_DRIVE_ROOT'] = DRIVE_ROOT
os.environ['H3_PERSIST_MODELS'] = '1' if PERSIST_MODELS_TO_DRIVE else '0'
os.environ['H3_PERSIST_OUTPUT'] = '1' if PERSIST_OUTPUT_TO_DRIVE else '0'
os.environ['H3_VRAM_MODE'] = 'auto'
os.environ['H3_RESERVE_VRAM_GB'] = H3_RESERVE_VRAM_GB
os.environ['H3_PREVIEW_METHOD'] = 'none'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

!bash install_comfy_h3.sh

CUSTOM='/content/ComfyUI/custom_nodes'
def clone_or_pull(url, folder):
    path=f'{CUSTOM}/{folder}'
    if os.path.isdir(path+'/.git'):
        subprocess.run(['git','-C',path,'pull','--ff-only'], check=False)
    else:
        subprocess.run(['git','clone','--depth','1',url,path], check=True)
    req=os.path.join(path,'requirements.txt')
    if os.path.exists(req):
        subprocess.run([sys.executable,'-m','pip','install','-r',req], check=False)

clone_or_pull('https://github.com/AIMixer/ComfyUI_MiniMaxH3_Director.git','ComfyUI_MiniMaxH3_Director')
clone_or_pull('https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git','ComfyUI-VideoHelperSuite')
clone_or_pull('https://github.com/kijai/ComfyUI-KJNodes.git','ComfyUI-KJNodes')
clone_or_pull('https://github.com/pixaroma/ComfyUI-Pixaroma.git','ComfyUI-Pixaroma')
subprocess.run([sys.executable,'-m','pip','install','-U','huggingface_hub'], check=True)
print('✅ ComfyUI + H3 custom nodes ready.')


## 5. Load Cloudflare secret + install cloudflared

Cloudflare Published Application must be:

- Hostname: `comfy.zetbros.com`
- Service: `http://127.0.0.1:8188`

The Colab secret must be named `CF_TUNNEL_TOKEN`.


In [ ]:
from google.colab import userdata
import os, re, platform, subprocess

raw = userdata.get('CF_TUNNEL_TOKEN')
if not raw:
    raise RuntimeError('CF_TUNNEL_TOKEN is missing or notebook access is disabled.')

m = re.search(r'(eyJ[A-Za-z0-9._-]+)', raw.strip())
if not m:
    raise RuntimeError('Could not find an eyJ... Cloudflare Tunnel token in CF_TUNNEL_TOKEN.')

os.environ['CF_TUNNEL_TOKEN_RAW'] = m.group(1)
print('✅ Tunnel token recognized. Token is intentionally not printed.')

arch = platform.machine().lower()
cf_arch = 'amd64' if arch in ('x86_64','amd64') else 'arm64' if arch in ('aarch64','arm64') else None
if not cf_arch:
    raise RuntimeError(f'Unsupported architecture: {arch}')

if subprocess.run(['bash','-lc','command -v cloudflared >/dev/null 2>&1']).returncode != 0:
    url = f'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-{cf_arch}'
    subprocess.run(['curl','-fL','--retry','3','--retry-delay','2',url,'-o','/usr/local/bin/cloudflared'], check=True)
    subprocess.run(['chmod','0755','/usr/local/bin/cloudflared'], check=True)

print(subprocess.check_output(['cloudflared','--version'], text=True).strip())


## 6. Start ComfyUI directly and verify the local origin

This deliberately does **not** use `launch_comfy_cloudflare.sh`. It mirrors the successful isolation test: start one process directly, then prove the local origin is healthy before Cloudflare starts.


In [ ]:
import os, subprocess, time
from pathlib import Path

LOG_DIR = Path('/content/h3_main_logs')
LOG_DIR.mkdir(parents=True, exist_ok=True)
COMFY_ROOT = '/content/ComfyUI'
PORT = 8188

# Remove anything from earlier tunnel tests or older Comfy launches in this runtime.
subprocess.run("pkill -f 'cf_origin_test_server.py'", shell=True, check=False)
subprocess.run("pkill -f 'python.*main.py.*--port 8188'", shell=True, check=False)
time.sleep(1)

COMFY_CMD = [
    sys.executable, 'main.py',
    '--listen', '127.0.0.1',
    '--port', str(PORT),
    '--disable-auto-launch',
    '--reserve-vram', str(H3_RESERVE_VRAM_GB),
    '--preview-method', 'none',
]

def start_comfy():
    log_handle = open(LOG_DIR/'comfyui.log', 'w')
    proc = subprocess.Popen(
        COMFY_CMD, cwd=COMFY_ROOT, stdout=log_handle, stderr=subprocess.STDOUT,
        env=os.environ.copy()
    )
    (LOG_DIR/'comfyui.pid').write_text(str(proc.pid))
    print('ComfyUI PID:', proc.pid)

    for _ in range(120):
        if proc.poll() is not None:
            raise RuntimeError('ComfyUI exited early. Check /content/h3_main_logs/comfyui.log')
        r = subprocess.run(
            ['curl','-fsS','--connect-timeout','2','--max-time','3','http://127.0.0.1:8188/system_stats'],
            stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
        )
        if r.returncode == 0:
            print('✅ Local ComfyUI origin is HTTP 200 at 127.0.0.1:8188')
            return proc
        time.sleep(2)
    raise RuntimeError('ComfyUI did not become ready in time.')

comfy_proc = start_comfy()
subprocess.run("ss -ltnp | grep ':8188' || true", shell=True)


## 7. Start Cloudflare Tunnel directly

This uses the same `subprocess.Popen` pattern that worked in the origin-isolation test.


In [ ]:
import os, subprocess, time
from pathlib import Path

LOG_DIR = Path('/content/h3_main_logs')
subprocess.run("pkill -f 'cloudflared.*tunnel.*run'", shell=True, check=False)
time.sleep(1)

cf_log_handle = open(LOG_DIR/'cloudflared.log', 'w')
CF_CMD = [
    'cloudflared','tunnel','--no-autoupdate','--loglevel','debug','run',
    '--token', os.environ['CF_TUNNEL_TOKEN_RAW']
]
cf_proc = subprocess.Popen(CF_CMD, stdout=cf_log_handle, stderr=subprocess.STDOUT)
(LOG_DIR/'cloudflared.pid').write_text(str(cf_proc.pid))
print('cloudflared PID:', cf_proc.pid)

for _ in range(45):
    time.sleep(1)
    if cf_proc.poll() is not None:
        raise RuntimeError('cloudflared exited early. Check /content/h3_main_logs/cloudflared.log')
    text = (LOG_DIR/'cloudflared.log').read_text(errors='replace')
    if 'Registered tunnel connection' in text:
        print('✅ Cloudflare Tunnel connector registered.')
        break
else:
    print('⚠️ No registration line seen yet. Inspect diagnostics below.')

print('🌐 Open now: https://comfy.zetbros.com')
print('Do this BEFORE downloading the H3 models. The bare ComfyUI page should load.')


## 8. Download H3 models while ComfyUI stays online

Only run this after `https://comfy.zetbros.com` successfully shows ComfyUI.


In [ ]:
from huggingface_hub import hf_hub_download
from pathlib import Path
import shutil

MODEL_ROOT = Path(f'{DRIVE_ROOT}/models' if PERSIST_MODELS_TO_DRIVE else '/content/ComfyUI/models')
for folder in ['diffusion_models','text_encoders','vae','loras','latent_upscale_models','upscale_models']:
    (MODEL_ROOT/folder).mkdir(parents=True, exist_ok=True)

if PERSIST_MODELS_TO_DRIVE:
    for folder in ['diffusion_models','text_encoders','vae','loras','latent_upscale_models','upscale_models']:
        local = Path('/content/ComfyUI/models')/folder
        drive_dir = MODEL_ROOT/folder
        drive_dir.mkdir(parents=True, exist_ok=True)
        if local.is_symlink():
            local.unlink()
        elif local.exists():
            shutil.rmtree(local) if local.is_dir() else local.unlink()
        local.symlink_to(drive_dir, target_is_directory=True)

downloads = [
    ('Comfy-Org/MiniMax-H3','diffusion_models/minimax_h3_fl2va_pruned_int8_convrot.safetensors',MODEL_ROOT,MODEL_ROOT/'diffusion_models'/'minimax_h3_fl2va_pruned_int8_convrot.safetensors'),
    ('Comfy-Org/MiniMax-H3','diffusion_models/minimax_h3_ref2va_pruned_int8_convrot.safetensors',MODEL_ROOT,MODEL_ROOT/'diffusion_models'/'minimax_h3_ref2va_pruned_int8_convrot.safetensors'),
    ('Comfy-Org/MiniMax-H3','text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors',MODEL_ROOT,MODEL_ROOT/'text_encoders'/'qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors'),
    ('Comfy-Org/MiniMax-H3','vae/minimax_h3_video_vae_fp16.safetensors',MODEL_ROOT,MODEL_ROOT/'vae'/'minimax_h3_video_vae_fp16.safetensors'),
    ('Comfy-Org/MiniMax-H3','vae/minimax_h3_audio_vae_fp32.safetensors',MODEL_ROOT,MODEL_ROOT/'vae'/'minimax_h3_audio_vae_fp32.safetensors'),
    ('lightx2v/Minimax-h3-Turbo','minimax_h3_ref2v_turbo_8step_v1.0_768p_comfyui_bf16.safetensors',MODEL_ROOT/'loras',MODEL_ROOT/'loras'/'minimax_h3_ref2v_turbo_8step_v1.0_768p_comfyui_bf16.safetensors'),
    ('lightx2v/Minimax-h3-Turbo','minimax_h3_fl2v_turbo_8step_v1.0_comfyui_bf16.safetensors',MODEL_ROOT/'loras',MODEL_ROOT/'loras'/'minimax_h3_fl2v_turbo_8step_v1.0_comfyui_bf16.safetensors'),
    ('LBH-123-AI/Minimax_h3_latent_Upscaler','minimax_h3_latent_upscaler_3d_fp16.safetensors',MODEL_ROOT/'latent_upscale_models',MODEL_ROOT/'latent_upscale_models'/'minimax_h3_latent_upscaler_3d_fp16.safetensors'),
]

for repo_id, filename, local_dir, target in downloads:
    if target.exists() and target.stat().st_size > 1024*1024:
        print('SKIP', target.name)
        continue
    print('DOWNLOAD', filename)
    hf_hub_download(repo_id=repo_id, filename=filename, local_dir=str(local_dir))
    if not target.exists():
        raise FileNotFoundError(target)
    print('READY', target.name)

print('✅ H3 model stack downloaded.')
print('Cloudflare stays connected during this download.')


## 9. Restart only ComfyUI so the model lists refresh

Cloudflare is deliberately left running. `comfy.zetbros.com` may show a short 502 only while ComfyUI is restarting, then it should recover automatically.


In [ ]:
import subprocess, time

subprocess.run("pkill -f 'python.*main.py.*--port 8188'", shell=True, check=False)
time.sleep(2)
comfy_proc = start_comfy()

# Prove Cloudflare was not restarted.
if cf_proc.poll() is None:
    print('✅ Existing Cloudflare connector is still running.')
else:
    raise RuntimeError('Cloudflare connector stopped unexpectedly. Rerun Section 7 only.')

print('✅ FINAL READY')
print('🌐 https://comfy.zetbros.com')
print('Refresh the browser and load your workflow manually.')


## Diagnostics

If the public page ever shows 502, refresh it once and immediately run this cell. It does not print the tunnel token.


In [ ]:
from pathlib import Path
import subprocess

LOG_DIR = Path('/content/h3_main_logs')
print('=== LOCAL COMFY ===')
subprocess.run("curl -sS -o /dev/null -w 'HTTP %{http_code}\n' --max-time 5 http://127.0.0.1:8188/system_stats || true", shell=True)
subprocess.run("ss -ltnp | grep ':8188' || true", shell=True)

print('\n=== PROCESS STATE ===')
print('ComfyUI running:', comfy_proc.poll() is None if 'comfy_proc' in globals() else 'unknown')
print('cloudflared running:', cf_proc.poll() is None if 'cf_proc' in globals() else 'unknown')

print('\n=== CLOUDFLARED LOG (last 160 lines) ===')
if (LOG_DIR/'cloudflared.log').exists():
    lines = (LOG_DIR/'cloudflared.log').read_text(errors='replace').splitlines()
    print('\n'.join(lines[-160:]))

print('\n=== COMFY LOG (last 120 lines) ===')
if (LOG_DIR/'comfyui.log').exists():
    lines = (LOG_DIR/'comfyui.log').read_text(errors='replace').splitlines()
    print('\n'.join(lines[-120:]))
